In [3]:
# Repository and artifact inspection

import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# 1. Project root discovery (same pattern as NB11 / NB12)
# -----------------------------------------------------------------------------
CURRENT_DIR = Path.cwd().resolve()

def discover_project_root():
    for p in (CURRENT_DIR, *CURRENT_DIR.parents):
        if (p / "nb08_artifacts").is_dir() and (p / "data" / "raw").is_dir():
            return p
    for p in (CURRENT_DIR, *CURRENT_DIR.parents):
        if (p / "nb08_artifacts").is_dir():
            return p
    return None

PROJECT_ROOT = discover_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from {CURRENT_DIR}. "
        "Expected a parent directory containing 'nb08_artifacts/'."
    )

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
NB08_DIR     = PROJECT_ROOT / "nb08_artifacts"
NB09_DIR     = PROJECT_ROOT / "nb09_artifacts"
NB11_DIR     = PROJECT_ROOT / "nb11_artifacts"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
NB13_OUT_DIR = PROJECT_ROOT / "nb13_artifacts"
NB13_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("Cell 1 — Repository and artifact inspection")
print("=" * 72)
print(f"Python:              {sys.version.split()[0]}")

# -----------------------------------------------------------------------------
# 2. Existing NB13 placeholder check
# -----------------------------------------------------------------------------
print("\n" + "-" * 72)
print("Existing Notebook 13 placeholder")
print("-" * 72)

placeholder_hits = []
for root in (NOTEBOOKS_DIR, PROJECT_ROOT):
    if not root.is_dir():
        continue
    for nb in root.glob("*13*.ipynb"):
        if ".ipynb_checkpoints" in str(nb):
            continue
        if nb in [h[0] for h in placeholder_hits]:
            continue
        try:
            data = json.loads(nb.read_text(encoding="utf-8", errors="ignore"))
            n_cells = len(data.get("cells", []))
            populated = n_cells > 0
        except Exception:
            n_cells = -1
            populated = "unknown"
        placeholder_hits.append((nb, n_cells, populated))

if placeholder_hits:
    for nb, n_cells, populated in placeholder_hits:
        size = nb.stat().st_size
        print(f"  FOUND: {nb.relative_to(PROJECT_ROOT)}  "
              f"({size:,} bytes, cells={n_cells}, populated={populated})")
    print("  → Do NOT overwrite a populated placeholder; create a new file if needed.")
else:
    print("  None found. A fresh notebook 13_deployment_feasibility.ipynb can be created.")

# -----------------------------------------------------------------------------
# 3. Required artifact presence
# -----------------------------------------------------------------------------
print("\n" + "-" * 72)
print("Required artifacts")
print("-" * 72)

REQUIRED = {
    # Frozen model core
    "TCN checkpoint":                 NB08_DIR / "tcn_preliminary" / "tcn_autoencoder_best.pth",
    "Frozen thresholds":              NB08_DIR / "frozen_thresholds.json",
    "Scaler":                         NB08_DIR / "scaler.joblib",
    "Evaluation summary":             NB08_DIR / "evaluation_summary.csv",
    "Final sequence scores":          NB08_DIR / "final_sequence_scores.npz",
    "Final observation scores":       NB08_DIR / "final_observation_scores.npz",
    "Evaluation sequences":           NB08_DIR / "evaluation_sequences.npz",
    "Attack evaluation sequences":    NB08_DIR / "attack_evaluation_sequences.npz",
    # NB09 event-level evidence (Test E input)
    "NB09 scenario detection":        NB09_DIR / "scenario_detection.csv",
    "NB09 scenario detection summary": NB09_DIR / "scenario_detection_summary.csv",
    "NB09 detection delays":          NB09_DIR / "detection_delays.csv",
    "NB09 detection delay summary":   NB09_DIR / "detection_delay_summary.csv",
    "NB09 validation false alarm summary": NB09_DIR / "validation_false_alarm_summary.csv",
    "NB09 validation false alarm periods": NB09_DIR / "validation_false_alarm_periods.csv",
    "NB09 fp_sensor_attribution":     NB09_DIR / "fp_sensor_attribution.csv",
    # NB11 report
    "NB11 final report":              NB11_DIR / "NB11_FINAL_REPORT.md",
    # Raw data (Test C replay)
    "Normal CSV":                     DATA_RAW_DIR / "normal.csv",
    "Attack CSV":                     DATA_RAW_DIR / "attack.csv",
}

inventory_rows = []
for label, path in REQUIRED.items():
    present = path.is_file()
    size = path.stat().st_size if present else None
    inventory_rows.append({
        "label": label, "path": str(path.relative_to(PROJECT_ROOT)),
        "present": present, "size_bytes": size,
    })
    marker = "✓" if present else "✗"
    size_str = f"{size:>12,}" if size else " " * 12
    print(f"  {marker} {label:36s}  {size_str}  {path.relative_to(PROJECT_ROOT)}")

missing = [r for r in inventory_rows if not r["present"]]
print(f"\n  Present: {len(inventory_rows) - len(missing)} / {len(inventory_rows)}")
if missing:
    print("  MISSING:")
    for r in missing:
        print(f"    - {r['label']}: {r['path']}")

# -----------------------------------------------------------------------------
# 4. Frozen threshold — exact value and NB11 cross-check
# -----------------------------------------------------------------------------
print("\n" + "-" * 72)
print("Frozen threshold — exact stored value vs NB11 citation")
print("-" * 72)

NB11_CITED_TCN_THRESHOLD = 0.003906734725460  # NB11 Cell 8 check #4 evidence

frozen_path = NB08_DIR / "frozen_thresholds.json"
if frozen_path.is_file():
    frozen = json.loads(frozen_path.read_text(encoding="utf-8"))
    tcn_thr = float(frozen["thresholds"]["tcn"])
    print(f"  Thresholds present for: {sorted(frozen['thresholds'].keys())}")
    print(f"  TCN threshold (exact):  {tcn_thr!r}")
    print(f"  TCN procedure:          {frozen['sources']['tcn']}")
    print(f"  Learned percentile:     {frozen['learned_percentile']}")
    print(f"  Learned interpolation:  {frozen['learned_interpolation']}")
    print(f"  n_threshold_sequences:  {frozen.get('n_threshold_sequences')}")
    print(f"  n_validation_sequences: {frozen.get('n_validation_sequences')}")
    print(f"  n_attack_sequences:     {frozen.get('n_attack_sequences')}")

    delta = abs(tcn_thr - NB11_CITED_TCN_THRESHOLD)
    print(f"\n  NB11 citation:          {NB11_CITED_TCN_THRESHOLD!r}")
    print(f"  |delta|:                {delta:.3e}")
    print(f"  Verdict:                "
          f"{'PASS — matches NB11' if delta < 1e-10 else 'ISSUE — differs from NB11'}")
else:
    print("  MISSING — cannot verify threshold")

# -----------------------------------------------------------------------------
# 5. Scaler — type + 42 features
# -----------------------------------------------------------------------------
print("\n" + "-" * 72)
print("Scaler — type + feature count")
print("-" * 72)

scaler_path = NB08_DIR / "scaler.joblib"
if scaler_path.is_file():
    import joblib
    scaler = joblib.load(scaler_path)
    print(f"  Type:               {type(scaler).__name__}")
    if hasattr(scaler, "feature_names_in_"):
        features = list(scaler.feature_names_in_)
        print(f"  Feature count:      {len(features)}")
        print(f"  First 5:            {features[:5]}")
        print(f"  Last  5:            {features[-5:]}")
        print(f"  Expected 42:        {'PASS' if len(features) == 42 else 'ISSUE'}")
    else:
        print(f"  n_features_in_:     {scaler.n_features_in_}")
else:
    print("  MISSING — cannot verify scaler")

# -----------------------------------------------------------------------------
# 6. TCN checkpoint — size + state-dict summary
# -----------------------------------------------------------------------------
print("\n" + "-" * 72)
print("TCN checkpoint — size + state_dict")
print("-" * 72)

NB12_RECORDED_CHECKPOINT_BYTES = 172_469
NB11_RECORDED_PARAM_COUNT       = 40_554

tcn_ckpt = NB08_DIR / "tcn_preliminary" / "tcn_autoencoder_best.pth"
if tcn_ckpt.is_file():
    size = tcn_ckpt.stat().st_size
    print(f"  Path:               {tcn_ckpt.relative_to(PROJECT_ROOT)}")
    print(f"  Size:               {size:,} bytes")
    print(f"  NB12 recorded:      {NB12_RECORDED_CHECKPOINT_BYTES:,} bytes")
    print(f"  Size matches NB12:  "
          f"{'PASS' if size == NB12_RECORDED_CHECKPOINT_BYTES else 'ISSUE'}")

    try:
        import torch
        state = torch.load(tcn_ckpt, map_location="cpu", weights_only=True)
        if isinstance(state, dict) and "model_state_dict" in state:
            print("  NOTE: checkpoint is a wrapper dict; extracting state_dict")
            state = state["model_state_dict"]
        n_params = sum(v.numel() for v in state.values() if hasattr(v, "numel"))
        print(f"  State-dict keys:    {len(state)}")
        print(f"  Parameter count:    {n_params:,}")
        print(f"  NB11 recorded:      {NB11_RECORDED_PARAM_COUNT:,}")
        print(f"  Param count matches:{' PASS' if n_params == NB11_RECORDED_PARAM_COUNT else ' ISSUE'}")
        print(f"  First few keys:     {list(state.keys())[:3]}")
    except ImportError:
        print("  PyTorch not available — cannot inspect state_dict")
    except Exception as e:
        print(f"  Could not load state_dict: {e}")
else:
    print("  MISSING — cannot verify checkpoint")

# -----------------------------------------------------------------------------
# 7. Sequence array shapes
# -----------------------------------------------------------------------------
print("\n" + "-" * 72)
print("Sequence array shapes")
print("-" * 72)

if (NB08_DIR / "evaluation_sequences.npz").is_file():
    with np.load(NB08_DIR / "evaluation_sequences.npz") as seq:
        print(f"  X_threshold_seq:        {seq['X_threshold_seq'].shape}   "
              f"(expected (47408, 30, 42))")
        print(f"  X_validation_seq:       {seq['X_validation_seq'].shape}  "
              f"(expected (79031, 30, 42))")
        print(f"  threshold_end_indices:  {seq['threshold_end_indices'].shape}")
        print(f"  validation_end_indices: {seq['validation_end_indices'].shape}")

if (NB08_DIR / "attack_evaluation_sequences.npz").is_file():
    with np.load(NB08_DIR / "attack_evaluation_sequences.npz") as atk:
        print(f"  X_attack_seq:           {atk['X_attack_seq'].shape}     "
              f"(expected (54592, 30, 42))")
        print(f"  attack_end_indices:     {atk['attack_end_indices'].shape}")

# -----------------------------------------------------------------------------
# 8. Git status (best-effort, non-blocking)
# -----------------------------------------------------------------------------
print("\n" + "-" * 72)
print("Git status (informational only)")
print("-" * 72)
try:
    result = subprocess.run(
        ["git", "status", "--short"],
        cwd=PROJECT_ROOT, capture_output=True, text=True, timeout=10,
    )
    if result.returncode == 0:
        lines = [ln for ln in result.stdout.splitlines() if ln.strip()]
        if lines:
            print(f"  {len(lines)} modified/untracked paths (first 10 shown):")
            for ln in lines[:10]:
                print(f"    {ln}")
        else:
            print("  Working tree clean.")
    else:
        print("  git not available or not a git repository.")
except Exception:
    print("  git not available or not a git repository.")

# -----------------------------------------------------------------------------
# 9. Persist inventory
# -----------------------------------------------------------------------------
inv_path = NB13_OUT_DIR / "cell1_artifact_inventory.csv"
pd.DataFrame(inventory_rows).to_csv(inv_path, index=False)
print(f"\nInventory saved to: {inv_path.relative_to(PROJECT_ROOT)}")

# -----------------------------------------------------------------------------
# 10. Verdict — is Test A unblocked?
# -----------------------------------------------------------------------------
print("\n" + "=" * 72)
print("Verdict and first prerequisite check")
print("=" * 72)
if not missing:
    print("  All required artifacts present.")
    print("  First prerequisite check after Cell 1:")
    print("    → Test A — load the exact selected TCN checkpoint and verify")
    print("      that its state_dict loads cleanly into a freshly-built")
    print("      TCNAutoencoder(n_features=42, channels=[32,32,32,32],")
    print("      kernel_size=3, dilations=[1,2,4,8]) reconstructed from NB08,")
    print("      then verify eval-mode inference produces finite, correctly")
    print("      shaped output for a small input batch.")
    print("  This is the gate for Test B (score parity) and every later test.")
else:
    print(f"  {len(missing)} artifact(s) missing. Test A blocked until resolved.")
print("=" * 72)

Cell 1 — Repository and artifact inspection
Python:              3.11.14

------------------------------------------------------------------------
Existing Notebook 13 placeholder
------------------------------------------------------------------------
  FOUND: notebooks\13_deployment_feasibility_and_historical_replay_rest.ipynb  (24,516 bytes, cells=2, populated=True)
  → Do NOT overwrite a populated placeholder; create a new file if needed.

------------------------------------------------------------------------
Required artifacts
------------------------------------------------------------------------
  ✓ TCN checkpoint                             172,469  nb08_artifacts\tcn_preliminary\tcn_autoencoder_best.pth
  ✓ Frozen thresholds                            1,347  nb08_artifacts\frozen_thresholds.json
  ✓ Scaler                                       2,127  nb08_artifacts\scaler.joblib
  ✓ Evaluation summary                           1,872  nb08_artifacts\evaluation_summary.csv
  

# Cell 1 — Repository and artifact inspection (result record)

## What was executed

Cell 1 discovered the project root, checked for an existing Notebook 13
placeholder, verified presence and size of 18 required artifacts, and
cross-checked the exact stored TCN threshold, scaler feature set,
TCN checkpoint size, and state-dict parameter count against the values
recorded by NB11 and NB12. It wrote `nb13_artifacts/cell1_artifact_inventory.csv`
and nothing else. No artifact was modified.

## What was actually observed

### Project root and placeholder

| Item | Observed |
|---|---|
| Project root discovered | Yes |
| Python | 3.11.14 |
| Existing notebook 13 | `notebooks/13_deployment_feasibility_and_historical_replay_rest.ipynb` (populated, 24,516 bytes in latest run) |

The NB13 placeholder already exists and is populated. Cell 1 did not
modify or overwrite it. Work continues in that file.

### Required artifacts — 18 / 18 present

All eighteen artifacts NB13 depends on are present with nonzero size:

- Frozen model core (7): TCN checkpoint (172,469 B), frozen_thresholds.json
  (1,347 B), scaler.joblib (2,127 B), evaluation_summary.csv (1,872 B),
  final_sequence_scores.npz (5,063,919 B), final_observation_scores.npz
  (2,927,982 B), evaluation_sequences.npz (10,735,316 B),
  attack_evaluation_sequences.npz (3,954,752 B)
- NB09 event-level evidence (7): scenario_detection.csv (9,923 B),
  scenario_detection_summary.csv (666 B), detection_delays.csv (15,121 B),
  detection_delay_summary.csv (410 B), validation_false_alarm_summary.csv
  (921 B), validation_false_alarm_periods.csv (620,301 B),
  fp_sensor_attribution.csv (1,281 B)
- NB11 final report (1): NB11_FINAL_REPORT.md (11,108 B)
- Raw data (2): normal.csv (402,374,633 B), attack.csv (15,162,438 B)

### Frozen threshold cross-check

| Field | Observed |
|---|---|
| TCN threshold (exact stored value) | `0.003906734725460411` |
| NB11 cited value | `0.00390673472546` |
| Absolute difference | `4.111e-16` |
| Verdict | PASS — matches NB11 |

The threshold stored in `frozen_thresholds.json` matches the value cited
by NB11 (Cell 8 check #4) to within floating-point precision. The
threshold procedure is recorded as *"99th pct of threshold partition
(interpolation=linear)"*, percentile = 99, and sequence counts for
threshold / validation / attack partitions are 47,408 / 79,031 / 54,592 —
all consistent with NB08's records.

### Scaler

| Field | Observed |
|---|---|
| Type | `MinMaxScaler` |
| Feature count | 42 |
| First 5 | `['FIT101', 'LIT101', 'MV101', 'P101', 'P102']` |
| Last 5 | `['PIT501', 'PIT502', 'PIT503', 'FIT601', 'P602']` |
| Verdict | PASS — 42 features |

A `InconsistentVersionWarning` was raised because the scaler was saved
under scikit-learn 1.6.1 and is being loaded under 1.9.1. This is the
same scikit-learn version drift documented by NB08 (limitation L11) and
NB12 (Cell 3 §version drift). It does not prevent the scaler from loading
or from transforming data, but any reconstitution of the training pipeline
would use different scikit-learn internals than the original run.

### TCN checkpoint

| Field | Observed |
|---|---|
| Size | 172,469 bytes |
| NB12 recorded size | 172,469 bytes |
| State-dict keys | 26 |
| Parameter count | 40,554 |
| NB11 recorded count | 40,554 |
| First keys | `encoder.0.conv1.conv.weight`, `encoder.0.conv1.conv.bias`, `encoder.0.conv2.conv.weight` |
| Verdict | PASS — size and param count match |

The state-dict is a bare `OrderedDict` (no wrapper dict). The key names
match NB08's `TCNBlock` / `CausalConv1d` module layout — the encoder uses
`conv1.conv.weight` / `conv2.conv.weight` and there are separate decoder
conv layers. This confirms the checkpoint corresponds to the
`TCNAutoencoder(n_features=42, channels=[32,32,32,32], kernel_size=3,
dilations=[1,2,4,8])` architecture defined in NB08 §7.1.

### Sequence array shapes

All three sequence arrays are present with the shapes NB08 and NB09
recorded:

| Array | Observed | Expected |
|---|---|---|
| `X_threshold_seq` | (47408, 30, 42) | (47408, 30, 42) |
| `X_validation_seq` | (79031, 30, 42) | (79031, 30, 42) |
| `X_attack_seq` | (54592, 30, 42) | (54592, 30, 42) |
| `threshold_end_indices` | (47408,) | (47408,) |
| `validation_end_indices` | (79031,) | (79031,) |
| `attack_end_indices` | (54592,) | (54592,) |

Window size is confirmed to be 30 (second axis of every sequence array).

### Git status

The second run of Cell 1 reported three modified/untracked paths:

- `M notebooks/08-hyperparameter-tuning-threshold-calibration.ipynb`
- `M notebooks/13_deployment_feasibility_and_historical_replay_rest.ipynb`
- `?? nb13_artifacts/`

Two observations worth recording:

1. **NB08 notebook source is modified in the working tree.** The first
   run of Cell 1 did not show this; between the two runs, NB08 was
   modified. This is a source-notebook change, not an artifact change.
   Cell 1 verifies artifact **presence and size** but not artifact
   **hashes** — the last hash verification was NB12's Cell 6 (15/15 NB08
   artifacts matched, 3/3 NB11 artifacts matched). Those hashes remain the
   authoritative evidence that the frozen artifacts are unmodified.
   Nevertheless, before committing the repository, the user should review
   what changed in the NB08 notebook source.
2. **`nb13_artifacts/` is untracked.** This is expected — Cell 1 created
   the directory and wrote `cell1_artifact_inventory.csv` into it. It
   should be added to `.gitignore` if the user does not want to commit
   Notebook 13's outputs.

## What this means

Every prerequisite for Cell 2 (Test A — load the exact selected model) is
satisfied:

- The selected model checkpoint is present, its size matches NB12's
  recorded size, and its state-dict structure and parameter count match
  NB11's recorded configuration.
- The scaler is loadable and exposes the expected 42 features in the
  expected order.
- The frozen threshold is the exact value NB11 cited (within floating-point
  precision) and is stored in an artifact that documents its procedure.
- All three sequence arrays exist with the shapes NB08 and NB09 recorded.
- Every NB09 artifact NB13 will need for Test E (event coverage and
  detection-delay comparison against the corrected NB09 replay) is
  present.

## Limitations that remain

- Scikit-learn version drift (1.6.1 → 1.9.1) is confirmed. This is a
  reproducibility packaging issue already documented by NB08 L11. It does
  not block NB13's replay tests because NB13 does not retrain — but any
  comparison between newly generated scores and NB08's frozen scores must
  remain within tolerance and must be reported as hardware/software-dependent
  rather than exact.
- Artifact *hashes* were not re-verified in Cell 1. The last verification
  is NB12's Cell 6. If the user commits the repository, those recorded
  hashes in `nb12_artifacts/cell6_artifact_inventory.csv` remain the
  authoritative reference.
- Git shows the NB08 notebook source is currently modified. This is worth
  reviewing before commit; it is separate from any artifact integrity
  question.

## Next step

Cell 2 (Test A) will rebuild `TCNAutoencoder` verbatim from NB08 §7.1,
load the checkpoint with `strict=True`, set the model to eval mode, run a
small inference pass against four real threshold sequences under
`torch.no_grad()`, and verify output shape, finite-ness, and score
vector shape. This is the gate for Test B (score parity).

In [ ]:
# Test A. Load the exact selected TCN model and verify inference.
import joblib
import torch.nn as nn
import torch.nn.functional as F

CURRENT_DIR = Path.cwd().resolve()

def discover_project_root():
    for p in (CURRENT_DIR, *CURRENT_DIR.parents):
        if (p / "nb08_artifacts").is_dir() and (p / "data" / "raw").is_dir():
            return p
    for p in (CURRENT_DIR, *CURRENT_DIR.parents):
        if (p / "nb08_artifacts").is_dir():
            return p
    return None

PROJECT_ROOT = discover_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate project root. Expected a parent directory "
        "containing nb08_artifacts/."
    )

NB08_DIR = PROJECT_ROOT / "nb08_artifacts"
NB13_OUT_DIR = PROJECT_ROOT / "nb13_artifacts"
NB13_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("Cell 2 — Test A: Load the exact selected TCN model")
print("=" * 72)

N_FEATURES = 42
WINDOW_SIZE = 30
CHANNELS = [32, 32, 32, 32]
KERNEL_SIZE = 3
DILATIONS = [1, 2, 4, 8]
EXPECTED_PARAM_COUNT = 40_554
EXPECTED_THRESHOLD = 0.003906734725460411

TCN_CHECKPOINT = NB08_DIR / "tcn_preliminary" / "tcn_autoencoder_best.pth"
SCALER_PATH = NB08_DIR / "scaler.joblib"
THRESHOLDS_PATH = NB08_DIR / "frozen_thresholds.json"
SEQUENCES_PATH = NB08_DIR / "evaluation_sequences.npz"

# Step 1. Rebuild TCNAutoencoder exactly as defined in NB08 Section 7.1.
# Verbatim, not simplified: this is the architecture the checkpoint was
# trained with.

class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0,
        )

    def forward(self, x):
        return self.conv(F.pad(x, (self.left_padding, 0)))


class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.conv1 = CausalConv1d(in_channels, out_channels, kernel_size, dilation)
        self.conv2 = CausalConv1d(out_channels, out_channels, kernel_size, dilation)
        self.activation = nn.ReLU()
        self.residual = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv1d(in_channels, out_channels, kernel_size=1)
        )

    def forward(self, x):
        residual = self.residual(x)
        x = self.activation(self.conv1(x))
        x = self.activation(self.conv2(x))
        return self.activation(x + residual)


class TCNAutoencoder(nn.Module):
    def __init__(self, n_features, channels, kernel_size, dilations):
        super().__init__()
        if len(channels) != len(dilations):
            raise ValueError("Each TCN block must have a dilation value.")

        encoder_layers = []
        in_channels = n_features
        for out_channels, dilation in zip(channels, dilations):
            encoder_layers.append(
                TCNBlock(in_channels, out_channels, kernel_size, dilation)
            )
            in_channels = out_channels
        self.encoder = nn.Sequential(*encoder_layers)

        decoder_layers = []
        reversed_channels = list(reversed(channels))
        reversed_dilations = list(reversed(dilations))
        for index, dilation in enumerate(reversed_dilations):
            current_channels = reversed_channels[index]
            next_channels = (
                reversed_channels[index + 1]
                if index + 1 < len(reversed_channels)
                else n_features
            )
            decoder_layers.append(
                CausalConv1d(current_channels, next_channels, kernel_size, dilation)
            )
            if index + 1 < len(reversed_dilations):
                decoder_layers.append(nn.ReLU())
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError("Expected input shape (batch, timesteps, features).")
        x = x.transpose(1, 2).contiguous()
        x = self.encoder(x)
        x = self.decoder(x)
        return x.transpose(1, 2).contiguous()


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device:               {DEVICE}")

# Step 2. Load scaler and confirm feature order matches the trained model.

scaler = joblib.load(SCALER_PATH)
if not hasattr(scaler, "feature_names_in_"):
    raise AttributeError("Scaler has no feature_names_in_; cannot verify feature order.")
FEATURES = list(scaler.feature_names_in_)
if len(FEATURES) != N_FEATURES:
    raise ValueError(
        f"Scaler has {len(FEATURES)} features, expected {N_FEATURES}."
    )

print(f"  Scaler:               {type(scaler).__name__}, {len(FEATURES)} features")
print(f"  First 5 features:     {FEATURES[:5]}")
print(f"  Last  5 features:     {FEATURES[-5:]}")

# Step 3. Load the frozen threshold from its authoritative saved artifact.

frozen = json.loads(THRESHOLDS_PATH.read_text(encoding="utf-8"))
tcn_threshold = float(frozen["thresholds"]["tcn"])
threshold_matches = abs(tcn_threshold - EXPECTED_THRESHOLD) < 1e-12
print(f"  TCN threshold stored: {tcn_threshold!r}")
print(f"  Threshold matches:    {'PASS' if threshold_matches else 'ISSUE'}")

# Step 4. Build the model and load the checkpoint with strict=True.

print("\n  Loading TCN checkpoint ...")
model = TCNAutoencoder(
    n_features=N_FEATURES,
    channels=CHANNELS,
    kernel_size=KERNEL_SIZE,
    dilations=DILATIONS,
).to(DEVICE)

try:
    state = torch.load(TCN_CHECKPOINT, map_location=DEVICE, weights_only=True)
except TypeError:
    state = torch.load(TCN_CHECKPOINT, map_location=DEVICE)

if isinstance(state, dict) and "model_state_dict" in state:
    state = state["model_state_dict"]

model.load_state_dict(state, strict=True)
model.eval()

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
params_match = param_count == EXPECTED_PARAM_COUNT
print(f"  Checkpoint size:      {TCN_CHECKPOINT.stat().st_size:,} bytes")
print(f"  State-dict keys:      {len(state)}")
print(f"  Parameter count:      {param_count:,}")
print(f"  Param count matches:  {'PASS' if params_match else 'ISSUE'}")

# Step 5. Confirm model is in eval mode.

is_eval = not model.training
print(f"\n  Model in eval mode:   {'PASS' if is_eval else 'ISSUE'}")

# Step 6. Smoke inference on a small batch of real threshold sequences.
# Uses torch.no_grad so no gradients are computed. Confirms output shape,
# finite-ness, and the score convention (mean over T and F of squared error).

with np.load(SEQUENCES_PATH) as seq:
    X_thresh = seq["X_threshold_seq"]

SMOKE_BATCH = 4
sample = torch.as_tensor(
    X_thresh[:SMOKE_BATCH].astype(np.float32),
    dtype=torch.float32,
    device=DEVICE,
)

with torch.no_grad():
    recon = model(sample)
    scores = ((recon - sample) ** 2).mean(dim=(1, 2))

shape_ok = tuple(recon.shape) == tuple(sample.shape)
finite_ok = bool(torch.isfinite(recon).all().item())
scores_shape_ok = tuple(scores.shape) == (SMOKE_BATCH,)
scores_finite_ok = bool(torch.isfinite(scores).all().item())
scores_nonneg = bool((scores >= 0).all().item())

print(f"  Input shape:          {tuple(sample.shape)}")
print(f"  Output shape:         {tuple(recon.shape)}")
print(f"  Output shape match:   {'PASS' if shape_ok else 'ISSUE'}")
print(f"  Output finite:        {'PASS' if finite_ok else 'ISSUE'}")
print(f"  Score vector shape:   {tuple(scores.shape)}")
print(f"  Score shape match:    {'PASS' if scores_shape_ok else 'ISSUE'}")
print(f"  Scores finite, >= 0:  "
      f"{'PASS' if (scores_finite_ok and scores_nonneg) else 'ISSUE'}")
print(f"  First 4 scores:       "
      f"{[round(float(x), 8) for x in scores.cpu().numpy()]}")

# Step 7. Persist Test A evidence to JSON for traceability.

all_pass = all([
    threshold_matches, params_match, is_eval,
    shape_ok, finite_ok, scores_shape_ok, scores_finite_ok, scores_nonneg,
])

record = {
    "test": "Test A — Load the exact selected model",
    "checkpoint_relative_path": str(TCN_CHECKPOINT.relative_to(PROJECT_ROOT)),
    "checkpoint_bytes": TCN_CHECKPOINT.stat().st_size,
    "scaler_relative_path": str(SCALER_PATH.relative_to(PROJECT_ROOT)),
    "scaler_type": type(scaler).__name__,
    "n_features": len(FEATURES),
    "features": FEATURES,
    "window_size": WINDOW_SIZE,
    "channels": CHANNELS,
    "kernel_size": KERNEL_SIZE,
    "dilations": DILATIONS,
    "param_count": int(param_count),
    "expected_param_count": EXPECTED_PARAM_COUNT,
    "threshold_stored": tcn_threshold,
    "threshold_matches_nb11": bool(threshold_matches),
    "model_in_eval_mode": bool(is_eval),
    "smoke_batch_size": SMOKE_BATCH,
    "smoke_output_shape": list(recon.shape),
    "smoke_output_finite": finite_ok,
    "smoke_scores": [float(x) for x in scores.cpu().numpy()],
    "device": str(DEVICE),
    "verdict": "PASS" if all_pass else "ISSUE",
}

out_path = NB13_OUT_DIR / "cell2_test_a.json"
out_path.write_text(json.dumps(record, indent=2), encoding="utf-8")

print("\n" + "=" * 72)
print(f"Test A verdict: {record['verdict']}")
print("=" * 72)
print(f"  Artifact: {out_path.relative_to(PROJECT_ROOT)}")
if record["verdict"] == "PASS":
    print("  Test A cleared.")
    print("  Next: Test B — score parity against saved NB08 scores.")
else:
    print("  Test A failed. Diagnose before proceeding to Test B.")